# Tesla source + voice handoff — NO VIDEO FILES

Upload **TESLA_CLAUDE_NO_VIDEO_HANDOFF.zip**. Source, master JSON, original plans, audits and frozen speech are included; no existing MP4 is supplied or downloaded. This notebook can rebuild the frozen base v2 at its original review resolution, using a device smoke test first. A full render can take substantial time. It does not implement the proposed premium v3/v4 re-cut or automatically apply the recorded final-camera repair. Read COLAB_AND_VOICE.md before use. Final actual inspection remains pending after any new render.


In [ ]:
from pathlib import Path
import hashlib, json, os, subprocess, sys, zipfile
from google.colab import files
uploaded = files.upload()
archives = [name for name in uploaded if name.lower().endswith('.zip')]
if len(archives) != 1: raise ValueError('Upload TESLA_CLAUDE_NO_VIDEO_HANDOFF.zip.')
archive = Path(archives[0])
PACKAGE_ROOT = Path('/content') / ('tesla_v2_' + hashlib.sha256(archive.read_bytes()).hexdigest()[:12])
PACKAGE_ROOT.mkdir(exist_ok=True)
with zipfile.ZipFile(archive) as bundle:
    if len(bundle.infolist()) > 2500 or sum(x.file_size for x in bundle.infolist()) > 2*1024**3:
        raise ValueError('Unexpected archive size.')
    for item in bundle.infolist():
        target = (PACKAGE_ROOT / item.filename).resolve()
        if not target.is_relative_to(PACKAGE_ROOT.resolve()) or ((item.external_attr >> 16) & 0o170000) == 0o120000:
            raise ValueError('Unsafe archive entry.')
    bundle.extractall(PACKAGE_ROOT)
HANDOFF_ROOT = PACKAGE_ROOT / 'TESLA_CLAUDE_NO_VIDEO_HANDOFF'
PACKAGE_ROOT = HANDOFF_ROOT / 'repository'
production = PACKAGE_ROOT / 'explainer/projects/tesla-hw3/build/full-v2'
manifest = json.loads((production / 'manifest.json').read_text())
def checked_clip_present(shot):
    folder = production / shot['id']
    if not (folder/'chapter.mp4').exists() or not (folder/'qa.json').exists(): return False
    qa = json.loads((folder/'qa.json').read_text())
    return qa.get('passed') and qa.get('video_sha256') == hashlib.sha256((folder/'chapter.mp4').read_bytes()).hexdigest()
all_clips_present = all(checked_clip_present(s) for s in manifest['shots'])
print('Complete clips bundled:', all_clips_present)


## Runtime

When all checked clips are present, this reassembles them with the current Python and FFmpeg. A separate Blender environment is installed only if rendering is needed.

In [ ]:
render_env = dict(os.environ, OMP_NUM_THREADS='2', OPENBLAS_NUM_THREADS='2')
if all_clips_present:
    RENDER_PYTHON = Path(sys.executable)
else:
    subprocess.run([sys.executable, str(PACKAGE_ROOT / 'explainer/colab/bootstrap.py')], cwd=PACKAGE_ROOT, check=True)
    RENDER_PYTHON = PACKAGE_ROOT / '.colab-render-venv/bin/python'
    USE_GPU = True  # Set False for deliberate CPU fallback.
    render_env['EXPLAINER_FORCE_CPU'] = '0' if USE_GPU else '1'
    subprocess.run([str(RENDER_PYTHON), '-m', 'explainer.projects.tesla-hw3.revision_colab', 'smoke'], cwd=PACKAGE_ROOT, env=render_env, check=True)


## Verify and resume

Frozen source, plan and narration hashes are checked before use. A changed device with existing native frames is rejected to avoid mixing frame versions. Completing the render leaves final inspection pending; it does not fabricate user approval.

In [ ]:
subprocess.run([str(RENDER_PYTHON), '-m', 'explainer.projects.tesla-hw3.revision_colab', 'resume'], cwd=PACKAGE_ROOT, env=render_env, check=True)


## Watch and download

Inspect the complete video and its transitions. Native 768×432 at 30fps, denoised and exported at 960×540. The JSON snapshot describes the package at creation; new output QA is in final-qa.json.

In [ ]:
from IPython.display import Video, display
display(Video(str(production / 'tesla-planning-v2.mp4'), embed=True, width=960))


In [ ]:
files.download(str(production / 'tesla-planning-v2.mp4'))


In [ ]:
files.download(str(production / 'tesla-project-v2.json'))
